In [1]:
# import libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import rasterio as rio
from rasterio.features import rasterize
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

In [2]:
# Data pipeline parameters

# Data directory - should contain a subdirectory for each site
# TODO hardcoded filepath
data_dir = '/network/scratch/m/matthew.fortier/shared/lichen/raw'

# File to put CSV data into
csv_file = 'dataset_file.csv'

# Classes can be commented out here if we wish to exclude them
class_map = {
    'lichen': 1,
    'rock': 4,
    'broadleaf': 5,
    'needleleaf': 6,
    'deadwood': 7,
    'graminoids': 8,
    'moss': 9,
    'soil': 10,
    'low_green': 12,
    'dry_branches': 13,
}

In [3]:
sites = os.listdir(data_dir)
print(sites)

['CG1-8A', 'CG1-8B', 'CS-103A', 'CS-103B', 'CS117B', 'CS3A', 'CS3B', 'CS3C', 'CS-46A', 'CS-46B', 'CS-59B', 'CS-96B', 'F3-19B', 'F3-19C', 'F3-20A', 'F3-20B', 'F3-20C', 'F3-8A', 'F3-8B', 'F3-8C', 'ZF20-11A', 'ZF46-15A', 'ZF46-37A', 'ZF46-45A']


In [4]:
def process_overlapping_shapes(label_gdf, chm_gdf):
    # Takes label and chm geometries, and returns a combined gdf
    # where each label is associated with a single canopy height
    combined_gdf = label_gdf.copy()
    combined_gdf['DN'] = -1
    chm_geometry = chm_gdf.geometry
    
    # iterate through each label shape
    for index, row in label_gdf.iterrows():
        intersecting_bools  = chm_geometry.intersects(row.geometry)
        intersecting_shapes = intersecting_bools[intersecting_bools != False]
        chm_values = list(chm_gdf.iloc[intersecting_shapes.index]['DN'])
        if len(chm_values) > 0:
            if row['class'] in [5, 6, 13]: # ['broadleaf', 'needleleaf', 'dry_branches']
                label_chm = max(chm_values)
            elif row['class'] in [1, 4, 7, 8, 9, 10, 12]: # ['lichen', 'rock', 'deadwood', 'graminoids', 'moss', 'soil', 'low_green']
                label_chm = min(chm_values)
            combined_gdf.loc[index, 'DN'] = label_chm
        else:
            combined_gdf.drop([index], inplace=True)
    return combined_gdf

In [5]:
def load_site_data(site: str, include_targeted_labels=False):
    # Retrieve class, RGB, canopy height, and certainty data from source files
    # Output them as a 6 layer raster (class, R, G, B, chm, certainty)
    
    rgb_file = os.path.join(data_dir, site, f'{site}_hp_transparent_mosaic_group1.tif')
    chm_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.tif')
    chm_shp_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.shp')
    label_file = os.path.join(data_dir, site, f'{site}_hp_labelledpoints.gpkg')

    # load RGB raster and metadata
    with rio.open(rgb_file) as f:
        # Get raster metadata
        height = f.height
        width = f.width
        crs = f.crs
        transform = f.transform
        output_raster = np.ndarray((6,height,width), dtype=np.uint8)
        rgb_raster = f.read((1,2,3), out=output_raster[1:4,:,:])
        
    # load shapefiles
    if include_targeted_labels == True:
        label_gdf_random = gpd.read_file(label_file).to_crs(crs).dropna(subset=['class', 'certainty'])
        label_gdf_targeted = gpd.read_file(label_file, layer=f'{site}_hp_targeted_labelling').to_crs(crs).dropna(subset=['class', 'certainty'])
        label_gdf = pd.concat([label_gdf_random, label_gdf_targeted], ignore_index=True)
    else:
        label_gdf = gpd.read_file(label_file).to_crs(crs).dropna(subset=['class', 'certainty'])
    label_gdf['certainty'] = label_gdf['certainty'].astype(np.uint8)
    chm_gdf = gpd.read_file(chm_shp_file).to_crs(crs)
    
    combined_gdf = process_overlapping_shapes(label_gdf, chm_gdf)
    
    rasterize(
        ((r['geometry'], r['class']) for _, r in combined_gdf.iterrows()),
        out_shape=(height, width),
        out=output_raster[0,:,:],
        transform=transform
    )
    
    rasterize(
        ((r['geometry'], r['DN']) for _, r in combined_gdf.iterrows()),
        out_shape=(height, width),
        out=output_raster[4,:,:],
        transform=transform
    )
    
    rasterize(
        ((r['geometry'], r['certainty']) for _, r in combined_gdf.iterrows()),
        out_shape=(height, width),
        out=output_raster[5,:,:],
        transform=transform
    )
    return output_raster

In [6]:
def convert_to_dataframe(combined_raster, site):
    # Get a list of indices where we have labelled pixels
    # Input is the combined raster as above
    # Returns dataframe of class pixels along with indices [ [348, 1451], [581, 2099], ... ]
    indices = np.argwhere(np.isin(combined_raster[0], list(class_map.values())))
    
    labels_array = combined_raster[0, indices[:,0], indices[:,1]].reshape(-1, 1)
    labels_df = pd.DataFrame(labels_array, columns=['veg_class'])
    
    rgb_features = combined_raster[1:4, indices[:,0], indices[:,1]].T
    rgb_df = pd.DataFrame(rgb_features, columns=['R', 'G', 'B'])
    
    chm_features = combined_raster[4, indices[:,0], indices[:,1]].reshape(-1, 1)
    chm_df = pd.DataFrame(chm_features, columns=['chm'])
    
    certainty_array = combined_raster[5, indices[:,0], indices[:,1]].reshape(-1, 1)
    certainty_df = pd.DataFrame(certainty_array, columns=['class_certainty'])
    
    site_df = pd.DataFrame({'site': [site] * len(labels_df), 'y_pos': indices[:,0], 'x_pos': indices[:,1]})
    
    df = pd.concat([site_df, labels_df, rgb_df, chm_df, certainty_df], axis=1)
    return df, indices

In [7]:
features = []
for site in sites:
    print(f'Processing {site}...')
    combined_raster = load_site_data(site)
    df, _ = convert_to_dataframe(combined_raster, site)
    features.append(df)

all_features = pd.concat(features, axis=0)

Processing CG1-8A...
Processing CG1-8B...
Processing CS-103A...
Processing CS-103B...
Processing CS117B...
Processing CS3A...
Processing CS3B...
Processing CS3C...
Processing CS-46A...
Processing CS-46B...
Processing CS-59B...
Processing CS-96B...
Processing F3-19B...
Processing F3-19C...
Processing F3-20A...
Processing F3-20B...
Processing F3-20C...
Processing F3-8A...
Processing F3-8B...
Processing F3-8C...
Processing ZF20-11A...
Processing ZF46-15A...
Processing ZF46-37A...
Processing ZF46-45A...


In [8]:
all_features

,site,y_pos,x_pos,veg_class,R,G,B,chm,class_certainty
0,CG1-8A,513,7948,13,224,222,219,1,2
1,CG1-8A,513,7949,13,233,227,222,1,2
2,CG1-8A,514,7948,13,223,220,218,1,2
3,CG1-8A,514,7949,13,228,221,215,1,2
4,CG1-8A,584,9232,12,93,107,72,1,2
...,...,...,...,...,...,...,...,...,...
1183,ZF46-45A,14786,2149,6,122,145,90,1,2
1184,ZF46-45A,14818,3243,6,86,107,69,2,2
1185,ZF46-45A,14818,3244,6,84,103,68,2,2
1186,ZF46-45A,14819,3243,6,90,109,76,2,2


In [9]:
# Some additional cleanup
df = all_features
# Normalize colors
for c in ['R', 'G', 'B']:
    df[c] = df[c].astype(float)/255.0

In [10]:
df.to_csv(csv_file, index=False)